In [1]:
#!/usr/bin/env python3
"""
diagnose_stripping_bug.py

Run this to figure out why some clusters keep full "cell_type::gene"
tokens instead of getting deduplicated down to bare gene symbols.
Edit the CONFIG block, then run and read the output.
"""
import re
import sys
import pandas as pd

# ============================================================================
# CONFIG -- edit these
# ============================================================================
NODES_FILE = '../11_hidef_results_faithful/heterogeneous_hidef_faithful.nodes'
BROKEN_CLUSTER_ID = 'Cluster1-0'    # one that showed unstripped members
WORKING_CLUSTER_ID = 'Cluster1-2'   # one that showed bare gene names


# ============================================================================
# CHECK 1: which hidef_multiplex_adapter.py is actually being imported?
# ============================================================================
print("=" * 80)
print("CHECK 1: import resolution")
print("=" * 80)
try:
    import hidef_multiplex_adapter
    print(f"hidef_multiplex_adapter loaded from: {hidef_multiplex_adapter.__file__}")
    print("If this path isn't the file you last edited, that's your bug --")
    print("restart your Python/notebook kernel (or del the module from sys.modules)")
    print("and re-import.")
except ImportError as e:
    print(f"Could not import hidef_multiplex_adapter: {e}")

if 'hidef_multiplex_adapter' in sys.modules:
    print(f"\nAlready in sys.modules at time of this run -- if you edited the file")
    print(f"AFTER first importing it in this session, Python is still using the")
    print(f"OLD cached version. Restart the kernel to be sure.")


# ============================================================================
# CHECK 2: raw .nodes row content, byte-for-byte, for a broken vs working cluster
# ============================================================================
print("\n" + "=" * 80)
print("CHECK 2: raw row content (repr() reveals hidden/non-ASCII characters)")
print("=" * 80)

df = pd.read_csv(NODES_FILE, sep='\t', header=None,
                  names=['cluster_id', 'size', 'members', 'persistence'],
                  dtype={'members': str})

for label, cid in [('BROKEN', BROKEN_CLUSTER_ID), ('WORKING', WORKING_CLUSTER_ID)]:
    matches = df[df['cluster_id'].astype(str) == str(cid)]
    if matches.empty:
        print(f"\n[{label}] cluster_id={cid!r} not found in {NODES_FILE}")
        continue
    row = matches.iloc[0]
    raw = str(row['members'])

    print(f"\n--- {label}: cluster_id={cid} ---")
    print(f"size column: {row['size']}   persistence column: {row['persistence']}")
    print(f"raw field length: {len(raw)} chars")
    print(f"first 300 chars (repr): {repr(raw[:300])}")

    toks = [t for t in re.split(r"[,\s]+", raw.strip()) if t]
    print(f"token count after split: {len(toks)}")
    print(f"first 5 tokens (repr): {[repr(t) for t in toks[:5]]}")

    looks_like_indices = all(t.lstrip("-").isdigit() for t in toks[:5])
    print(f"looks_like_indices (first 5 all-digit)?: {looks_like_indices}")

    if toks:
        first_tok = toks[0]
        has_double_colon = "::" in first_tok
        print(f"first token contains literal '::' (ASCII)?: {has_double_colon}")
        # scan for colon-like unicode lookalikes
        colonlike = [c for c in first_tok if c in (':', '\u2236', '\u2237', '\uff1a', '\u02d0')]
        if colonlike:
            print(f"colon-like characters found in first token: {[repr(c) for c in colonlike]}")


# ============================================================================
# CHECK 3: does the supra-graph pickle's node_names line up, if you're using it?
# ============================================================================
print("\n" + "=" * 80)
print("CHECK 3 (optional): supra-graph node_names sanity check")
print("=" * 80)
SUPRA_GRAPH_PKL = '../11_hidef_hetero/supra_graph.pkl'  # set this path if you pass supra_graph_pkl to the loader
if SUPRA_GRAPH_PKL:
    import pickle
    with open(SUPRA_GRAPH_PKL, 'rb') as f:
        g = pickle.load(f)
    print(f"supra-graph vertex count: {g.vcount()}")
    print(f"first 5 node names: {g.vs['name'][:5]}")
else:
    print("SUPRA_GRAPH_PKL not set -- skipping. Set it above if you pass "
          "supra_graph_pkl to load_multiplex_clusters/patch_analyzer_for_multiplex.")

print("\nDone. Paste the CHECK 1/2/3 output back and we can pinpoint the fix.")

CHECK 1: import resolution
hidef_multiplex_adapter loaded from: /lisc/home/user/gerardursin/multiplex/phen/hidef_multiplex_adapter.py
If this path isn't the file you last edited, that's your bug --
restart your Python/notebook kernel (or del the module from sys.modules)
and re-import.

Already in sys.modules at time of this run -- if you edited the file
AFTER first importing it in this session, Python is still using the
OLD cached version. Restart the kernel to be sure.

CHECK 2: raw row content (repr() reveals hidden/non-ASCII characters)

--- BROKEN: cluster_id=Cluster1-0 ---
size column: 17051   persistence column: 19
raw field length: 518009 chars
first 300 chars (repr): 'global_ppi::global_ppi::A1BG global_ppi::global_ppi::A1CF global_ppi::global_ppi::A2M global_ppi::global_ppi::A2ML1 global_ppi::global_ppi::A4GALT global_ppi::global_ppi::A4GNT global_ppi::global_ppi::AAAS global_ppi::global_ppi::AACS global_ppi::global_ppi::AADAC global_ppi::global_ppi::AADAT globa'
token count a

In [1]:
from enrichment_plots_multiplex import load_persistence_map
import pandas as pd

NODES_FILE = '../11_hidef_results_faithful_cpm/heterogeneous_hidef_faithful_cpm.nodes'
SUPRA_GRAPH_PKL = '../11_hidef_hetero/supra_graph.pkl'
SEED_GENES_FILE = '../00_inputs/seed_genes.csv'

pm = load_persistence_map(
    hidef_source='multiplex', nodes_file=NODES_FILE,
    supra_graph_pkl=SUPRA_GRAPH_PKL, seed_genes_file=SEED_GENES_FILE,
    persistence_threshold=0,
)
pd.DataFrame({'cluster_id': [f'HiDeF_Cluster_{k}' for k in pm.keys()],
              'persistence': list(pm.values())}).to_csv('cluster_persistence_lookup.csv', index=False)

[multiplex adapter] 1,582 clusters kept from ../11_hidef_results_faithful_cpm/heterogeneous_hidef_faithful_cpm.nodes (dropped 0 below persistence=0, 6,414 below min_size_after_dedup=2, 0 with unresolved index-form members)


In [2]:
#!/usr/bin/env python3
"""Stack every analyzer's cluster-level summary into one long table:
cluster_id, analysis, adjusted_p_value  (input for step11k)."""
import glob, os, sys
import pandas as pd
import pipeline_config as cfg

SUMMARY = {
    "proteomics": ("proteomics_analysis", "proteomics_summarized_results.csv"),
    "neuropathic_pain": ("neuropathic_pain_analysis", "neuropathic_pain_summarized_results.csv"),
    "orphanet": ("orphanet_enrichr_analysis", "orphanet_enrichr_summarized_results.csv"),
    "omim": ("omim_ps_analysis", "omim_ps_summarized_results.csv"),
    "phenotypes": ("phenotypes_analysis", "phenotype_summarized_results.csv"),
    "spatial": ("spatial_analysis", "spatial_transcriptomics_hidef_comparison_summarized_results.csv"),
}
base = os.path.join(cfg.OUTPUT_DIR, cfg.PPI_SOURCE)
frames = []
for name, (dpat, fname) in SUMMARY.items():
    hits = glob.glob(os.path.join(base, f"{dpat}_{cfg.CLUSTER_TYPE}*", fname))
    if not hits:
        print(f"[skip] {name}: no summary file found"); continue
    df = pd.read_csv(hits[0])
    if "adjusted_p_value" not in df.columns:
        print(f"[skip] {name}: no adjusted_p_value column"); continue
    frames.append(df[["cluster_id", "adjusted_p_value"]].assign(analysis=name))
    print(f"[ok]   {name}: {len(df)} clusters")
if not frames:
    sys.exit("nothing found")
out = "cluster_enrichment_long.csv"
pd.concat(frames).to_csv(out, index=False)
print(f"saved -> {out}")

[ok]   proteomics: 49 clusters
[ok]   neuropathic_pain: 87 clusters
[ok]   orphanet: 1556 clusters
[ok]   omim: 9600 clusters
[ok]   phenotypes: 114 clusters
[ok]   spatial: 516 clusters
saved -> cluster_enrichment_long.csv
